# 🎙️ Free Voice Clone Test (Chatterbox)

Clone **your own voice** and generate speech — free, no API key, runs on Google's GPU.

**First:** confirm Colab gave you a GPU — `Runtime → Change runtime type → T4 GPU`. (This notebook requests one automatically, but double-check.)

Then run each cell top to bottom with **Shift+Enter**.

What happens:
1. Installs Chatterbox (free, open-source voice cloning)
2. Downloads a 12-second sample of your AI voice
3. Clones it and reads a test sentence
4. Plays the result right here — you judge with your ears

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU — go to Runtime > Change runtime type > T4 GPU, then re-run this cell.")

In [ ]:
# Install Chatterbox (a couple of minutes the first time)
!pip install -q chatterbox-tts

In [ ]:
import os
import urllib.request

REF_URL = "https://muse.ai/files/766846519847631/2863599110677148/ynz9vztyaopu66ldy697cfo0/domi-ref.wav"
REF_PATH = "my-voice.wav"

if not os.path.exists(REF_PATH):
    try:
        print("Downloading your voice sample...")
        urllib.request.urlretrieve(REF_URL, REF_PATH)
        print("Voice sample ready.")
    except Exception as e:
        print("Download didn't work, you'll upload manually:", e)

if not os.path.exists(REF_PATH):
    from google.colab import files
    print("Upload a 5-15 second clip of clear speech (wav or mp3):")
    uploaded = files.upload()
    REF_PATH = list(uploaded.keys())[0]

print("Using voice reference:", REF_PATH)

In [ ]:
import torchaudio as ta
from chatterbox.tts import ChatterboxTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading model on", device, "(first run downloads ~1GB)...")
model = ChatterboxTTS.from_pretrained(device=device)

TEXT = "Everybody's going crazy over this new humanoid robot. People are saying it'll make women obsolete. Seems like guys are gonna desire this."

print("Generating your cloned voice...")
wav = model.generate(TEXT, audio_prompt_path=REF_PATH)
ta.save("my-voice-test.wav", wav, model.sr)
print("Done — hit play below and judge:")

from IPython.display import Audio
Audio("my-voice-test.wav")

## Try your own text

Change the sentence below, then run the cell — same cloned voice, new words.

In [ ]:
TEXT2 = "This is my cloned voice, reading whatever I type, for free."
wav2 = model.generate(TEXT2, audio_prompt_path=REF_PATH)
ta.save("my-voice-custom.wav", wav2, model.sr)
from IPython.display import Audio
Audio("my-voice-custom.wav")

In [ ]:
# Download the audio file to your own device
from google.colab import files
files.download("my-voice-test.wav")